# Import libraries

In [1]:
# 0. Import
import os
from re import search
from dfply import *
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns

In [2]:
import celloracle as co
co.__version__

/Users/argelagr/git/CellOracle/celloracle/trajectory/markov_simulation.py:17: NumbaWarning: 
Compilation is falling back to object mode WITH looplifting enabled because Function "_walk" failed type inference due to: Undecided type $22load_method.7 := <undecided>
During: resolving caller type: $22load_method.7
During: typing of call at /Users/argelagr/git/CellOracle/celloracle/trajectory/markov_simulation.py (40)


File "../../../../../../git/CellOracle/celloracle/trajectory/markov_simulation.py", line 40:
def _walk(start_cell_id_array, transition_prob, n_steps):
    <source elided>

    li.append(list(ids_now))
    ^

  @jit(i8[:,:](i8[:], f8[:,:], i8))
/Users/argelagr/git/CellOracle/celloracle/trajectory/markov_simulation.py:17: NumbaWarning: 
Compilation is falling back to object mode WITHOUT looplifting enabled because Function "_walk" failed type inference due to: Cannot determine Numba type of <class 'numba.core.dispatcher.LiftedLoop'>

File "../../../../../../git/CellOracle/cello

1 package does not meet CellOracle requirement.
 Your h5py version is 2.10.0. Please install h5py>=3.1.0


'0.8.0'

In [3]:
# visualization settings
%config InlineBackend.figure_format = 'retina'
%matplotlib inline

plt.rcParams['figure.figsize'] = [6, 4.5]
plt.rcParams["savefig.dpi"] = 300

# Load settings

In [4]:
if search("ricard", os.uname()[1]):
    exec(open('/Users/ricard/gastrulation_multiome_10x/settings.py').read())
    exec(open('/Users/ricard/gastrulation_multiome_10x/utils.py').read())
elif search("ebi", os.uname()[1]):
    exec(open('/homes/ricard/gastrulation_multiome_10x/settings.py').read())
    exec(open('/homes/ricard/gastrulation_multiome_10x/utils.py').read())
elif search("Workstation", os.uname()[1]):
    exec(open('/home/lijingyu/gastrulation/gastrulation_multiome_10x/settings.py').read())
    exec(open('/home/lijingyu/gastrulation/gastrulation_multiome_10x/utils.py').read())
else:
    exit("Computer not recognised")

## Define I/O

In [5]:
io["outdir"] = io["basedir"] + "/results/rna_atac/GRN/trajectories/blood_trajectory/"

NameError: name 'io' is not defined

In [ ]:
io['anndata'] = io["basedir"] + "/results/rna_atac/rna_vs_acc/trajectories/blood_trajectory/anndata.h5ad"
io['chip_GRN'] = io["basedir"] + '/results/rna_atac/GRN/chip_GRN.csv'
io['blood_chip_oracle'] = io['outdir'] + "/blood_chip.celloracle.oracle"
io['blood_chip_links'] = io['outdir'] + '/blood_chip.celloracle.links'

## Define options 

scanpy options

In [ ]:
# %%capture
# sc.settings.verbosity = 3
# sc.logging.print_versions()
sc.settings.set_figure_params(dpi=80, frameon=False, figsize=(8, 7), facecolor='white')

In [ ]:
opts["celltypes"] = [
   "Haematoendothelial_progenitors",
   "Blood_progenitors_1",
   "Blood_progenitors_2",
   "Erythroid1",
   "Erythroid2",
   "Erythroid3"
]

# 1. Prepare data



## 1.1. Load processed gene expression data (anndata)


In [ ]:
# Load data. !!Replace the data path below when you use another data.
adata = sc.read_h5ad(io["anndata"])

In [ ]:
adata.obs['trajectory'] = ['blood']*adata.n_obs

In [ ]:
adata.uns['trajectory_colors'] = np.array(['#D33F6A'])

In [ ]:
adata

In [ ]:
adata.layers["raw_count"] = adata.X.copy()

In [ ]:
sc.pp.scale(adata, max_value=10)

sc.tl.pca(adata, svd_solver='arpack')

sc.pl.pca(adata)

sc.pl.pca_variance_ratio(adata, log=True)

In [ ]:
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=25)
sc.tl.diffmap(adata)

In [ ]:
sc.pl.diffmap(adata,color=['celltype.mapped'])

In [ ]:
sc.pl.diffmap(adata,color=['DC1'])

In [ ]:
adata.obsm['Diffmap_embedding'] = adata.obsm['X_diffmap'][:,1:3]  

In [ ]:
# We use raw mRNA count as an input of Oracle object.
adata.X = adata.layers["raw_count"].copy()

## 1.2. Load TF data. 
For the GRN inference, celloracle needs TF information, which contains lists of the regulatory candidate genes. 

In [ ]:
TFinfo_df = pd.read_csv(io['chip_GRN'],header=0)

In [ ]:
TFinfo_df.shape

In [ ]:
TFinfo_df = TFinfo_df.rename(columns={'Unnamed: 0': 'peak_id','Target':'gene_short_name'})
TFinfo_df.head()

# 2. Initiate Oracle object

Celloracle has a custom called Oracle. We can use Oracle for the data preprocessing and GRN inference steps.
The Oracle object stores all of necessary information and does the calculations with its internal functions.
We instantiate an Oracle object, then input the gene expression data (anndata) and a TFinfo into the Oracle object.

In [ ]:
# Instantiate Oracle object
oracle = co.Oracle()

## 2.1. load gene expression data into oracle object.

In [ ]:
oracle.import_anndata_as_raw_count(adata=adata,
                                   cluster_column_name="trajectory",
                                   embedding_name="Diffmap_embedding")
# You can load TF info dataframe with the following code.


## 2.2. Load TFinfo into oracle object

In [ ]:
oracle.import_TF_data(TF_info_matrix=TFinfo_df)

# 3. Knn imputation
Celloracle uses almost the same strategy as velocyto for visualizing cell transitions. This process requires KNN imputation in advance.

For the KNN imputation, we need PCA and PC selection first.

## 3.1. PCA

In [ ]:
# Perform PCA
oracle.perform_PCA()

# Select important PCs
plt.plot(np.cumsum(oracle.pca.explained_variance_ratio_)[:100])
n_comps = np.where(np.diff(np.diff(np.cumsum(oracle.pca.explained_variance_ratio_))>0.002))[0][0]
plt.axvline(n_comps, c="k")
print(n_comps)
n_comps = min(n_comps, 50)

## 3.2. KNN imputation

Estimate the optimal number of nearest neighbors for KNN imputation.

In [ ]:
n_cell = oracle.adata.shape[0]
print(f"cell number is :{n_cell}")

In [ ]:
k = int(0.025*n_cell)
print(f"Auto-selected k is :{k}")

In [ ]:
oracle.knn_imputation(n_pca_dims=n_comps, k=k, balanced=True, b_sight=k*8,
                      b_maxl=k*4, n_jobs=4)

# 4. Save and Load.

Celloracle has some custom-classes: Links, Oracle and TFinfo.
You can save such an object using "to_hdf5".

Pleasae use "load_hdf5" function to load the file.


In [ ]:
# Save oracle object.
oracle.to_hdf5(io['blood_chip_oracle'])

# 5. GRN calculation
The next step is constructing a cluster-specific GRN for all clusters.

You can calculate  GRNs with the "get_links" function, and the function returns GRNs as a Links object.
The Links object stores inferred GRNs and the corresponding metadata. You can do network analysis with the Links object.

The GRN will be calculated for each cluster/sub-group.
In the example below, we construct GRN for each unit of the "trajectory" clustering.

The GRNs can be calculated at any arbitrary unit as long as the clustering information is stored in anndata.

## 5.1. Get GRNs

In [ ]:
links = oracle.get_links(cluster_name_for_GRN_unit="trajectory", alpha=10,
                        verbose_level=10, test_mode=False)
# Calculate GRN for each population in "GRN_unit" clustering unit.
# This step may take long time.

## 5.2. Export GRNs

Although celloracle has many functions for network analysis, you can analyze GRNs by hand if you choose.
The raw GRN data is stored in the attribute of "links_dict".

In [94]:
links.to_hdf5(io['blood_chip_links'])